# Lab Day 1 — Mạng nơ-ron và thí nghiệm huấn luyện (Forest CoverType)

**Sinh viên:** Nguyễn Văn Quốc Việt — **MSSV:** 2A202602973

Notebook nằm ở `submission_2A202602973/code/`. Dữ liệu và script ở `../../data`, `../../scripts`;
kết quả ghi vào `../figures/`, `../results/`, `../experiments.xlsx`, `../predictions_eval.csv`, `../eval_result.json`.

**Chạy trên Colab:** *Runtime → Change runtime type → T4 GPU*, rồi *Run all*. Ô đầu tiên tự mount Drive, clone/pull repo
và `cd` vào đúng thư mục `code/`.

Quy tắc: mọi lựa chọn cấu hình chỉ dựa trên **val**. Tập **eval** chỉ dùng ở Part 4 cho baseline và cấu hình cuối cùng.

In [ ]:
# ===== (Chỉ trên Colab) mount Drive, clone/pull repo, cd vào submission/code =====
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/vietvuivui/K4-Track4-Day1-NguyenVanQuocViet-2A202602973-Neuralnetwork.git"
    REPO_DIR = "/content/drive/MyDrive/lab1"          # lưu trên Drive để không mất kết quả khi Colab ngắt
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    %cd {REPO_DIR}/submission_2A202602973/code
print("cwd:", os.getcwd())

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, math, subprocess, importlib
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # = submission_<MSSV>/
sys.path.insert(0, os.getcwd())

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)
os.makedirs(f"{OUT_DIR}/extra", exist_ok=True)     # dự đoán eval của baseline (không phải file nộp chính)

import data, model, optimizer, train, plots, results_table
for m in (data, model, optimizer, train, plots, results_table):
    importlib.reload(m)                              # sửa .py rồi chạy lại ô này là đủ
from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval, compute_loss
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata (`scripts/split_data.py`), tách validation 20% **từ train** (phân tầng, seed 42),
chuẩn hoá 10 cột số bằng mean/std của phần train còn lại, đưa toàn bộ lên device một lần.

In [ ]:
if not os.path.exists(f"{REPO_ROOT}/data/processed/train.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

D = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert len(D["X_tr"]) == 371_847 and len(D["X_val"]) == 92_962 and len(D["X_eval"]) == 116_203

num = D["X_tr"][:, :10]
print("mean 10 cột số trên X_tr:", num.mean(0).cpu().numpy().round(4))
print("std  10 cột số trên X_tr:", num.std(0).cpu().numpy().round(4))
print("44 cột nhị phân chỉ có giá trị:", torch.unique(D["X_tr"][:, 10:]).tolist())

**Nhận xét Part 0:** phần train còn 371 847 mẫu, val 92 962, eval 116 203. Tỉ lệ lớp giữa ba tập gần như bằng nhau (nhờ phân tầng).
Sau chuẩn hoá, 10 cột số trên X_tr có mean ≈ 0, std ≈ 1. Accuracy "luôn đoán lớp đa số" trên val ≈ 0,4876: đây là mốc thấp nhất.
mean/std chỉ tính trên X_tr: nếu tính cả val/eval thì thống kê của dữ liệu đánh giá rò rỉ vào tiền xử lý.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
`M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model. Khởi tạo He (`kaiming_normal_`, bias = 0).

In [ ]:
# 1a-1b: shape và số tham số
set_seed(0)
m = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
print(m)
n_params = count_params(m)
print("số tham số:", n_params)
assert n_params == EXPECTED_PARAMS[(256, 128)] == 47_879

xb = torch.randn(8, 54, device=device)
h = xb
for layer in m.net:
    h = layer(h)
    print(f"  sau {layer.__class__.__name__:7s}: {tuple(h.shape)}")
assert m(xb).shape == (8, 7)
for hid in [(512, 256), (256, 128, 64)]:
    assert count_params(MLP(hidden=hid)) == EXPECTED_PARAMS[hid]
print("M-wide / M-deep cũng khớp số tham số quy định")

In [ ]:
# 1c-1: loss bước 0 trên val (eval mode) ≈ ln 7
step0 = evaluate(m, D["X_val"], D["y_val"], "ce")
print(f"loss bước 0 trên val = {step0['loss']:.4f}   (ln 7 = {math.log(7):.4f}, lệch {step0['loss'] - math.log(7):+.4f})")
print(f"accuracy bước 0 = {step0['acc']:.4f}")

In [ ]:
# 1c-2: quá khớp 20 mẫu (không dropout, Adam lr 1e-2, 500 bước)
set_seed(0)
m20 = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt20 = torch.optim.Adam(m20.parameters(), lr=1e-2)
x20, y20 = D["X_tr"][:20], D["y_tr"][:20]
losses20 = []
for step in range(500):
    m20.train()
    loss = compute_loss(m20(x20), y20, "ce")
    opt20.zero_grad(set_to_none=True); loss.backward(); opt20.step()
    losses20.append(loss.item())
acc20 = (predict(m20, x20) == y20).float().mean().item()
print(f"loss đầu = {losses20[0]:.4f}, loss cuối = {losses20[-1]:.6f}, accuracy trên 20 mẫu = {acc20:.2f}")
plt.figure(figsize=(6, 3.5)); plt.semilogy(losses20)
plt.title("Quá khớp 20 mẫu (M-base, Adam lr=1e-2)"); plt.xlabel("bước"); plt.ylabel("CE loss (log)"); plt.grid(alpha=.3)
plt.show()

In [ ]:
# 1d: gradient có chảy tới mọi tham số?
set_seed(0)
mg = MLP(hidden=(256, 128), init="he").to(device)
loss = compute_loss(mg(D["X_tr"][:512]), D["y_tr"][:512], "ce")
loss.backward()
for name, p in mg.named_parameters():
    g = p.grad
    print(f"  {name:14s} shape={tuple(p.shape)!s:12s} ‖grad‖ = {g.norm().item():.4e}")
    assert g is not None and g.norm().item() > 0, name
print("Mọi tham số đều có gradient khác 0.")

**Nhận xét Part 1 (điền sau khi chạy):** loss bước 0 đo được = ___ so với ln 7 = 1,946 — với khởi tạo He và bias = 0, logit ở bước 0 nhỏ
và gần như đối xứng giữa các lớp nên softmax ≈ đều 1/7, loss ≈ −ln(1/7). Quá khớp 20 mẫu: loss từ ___ về ___, accuracy 100% ⇒ pipeline
(nhãn, loss, backward, optimizer) đúng. Mọi tham số W1,b1,W2,b2,W3,b3 có gradient khác 0.

## Part 2 — Pipeline và baseline
Baseline: M-base, SGD + momentum 0,9, CE, He, batch 512, 20 epoch, không dropout/clip, FP32.

Các hàm tiện ích bên dưới: `run()` chạy một cấu hình, lưu `results/<exp_id>.json` và `figures/<exp_id>.png`.
Nếu Colab ngắt giữa chừng, chạy lại notebook sẽ **nạp lại** kết quả đã có thay vì huấn luyện lại (đặt `FORCE = True` để chạy lại tất cả).

In [ ]:
FORCE = False
RESULTS = {}

def run(cfg, force=False, verbose=False):
    cfg = {**DEFAULT_CFG, **cfg}
    path = f"{OUT_DIR}/results/{cfg['exp_id']}.json"
    if os.path.exists(path) and not (force or FORCE):
        with open(path, encoding="utf-8") as f:
            r = json.load(f)
        same = all(r["cfg"].get(k) == (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items())
        if same:
            RESULTS[cfg["exp_id"]] = r
            s = r["summary"]
            print(f"[nạp lại] {cfg['exp_id']:24s} val_F1={s['val_macro_f1']:.4f} val_acc={s['val_acc']:.4f}")
            return r
    r = run_experiment(cfg, D, verbose=verbose)
    save_result(r, f"{OUT_DIR}/results")
    plot_run(r, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")
    RESULTS[cfg["exp_id"]] = r
    s = r["summary"]
    print(f"[chạy]    {cfg['exp_id']:24s} val_F1={s['val_macro_f1']:.4f} val_acc={s['val_acc']:.4f} "
          f"best_ep={s['best_epoch']:2d} step0={s['step0_loss']:.3f} {s['time_per_epoch_s'] or 0:.1f}s/ep"
          + ("  DIVERGED" if s["diverged"] else ""))
    return r

def ensure_state(exp_id):
    # Cần best_state (trọng số) để dự đoán eval: nếu kết quả được nạp từ JSON thì huấn luyện lại đúng cấu hình đó.
    r = RESULTS[exp_id]
    if "best_state" not in r:
        cfg = dict(r["cfg"]); cfg["hidden"] = tuple(cfg["hidden"])
        r = run(cfg, force=True)
    return r

def table(ids, base_mean=None, two_sigma=None):
    rows = []
    for i in ids:
        r = RESULTS[i]; s = r["summary"]; c = r["cfg"]
        row = dict(exp_id=i, opt=c["optimizer"], lr=c["lr"], batch=c["batch"], hidden="-".join(map(str, c["hidden"])),
                   drop=c["dropout"], clip=c["clip_norm"], prec=c["precision"], init=c["init"],
                   step0=s["step0_loss"], best_ep=s["best_epoch"], best_val_loss=s["best_val_loss"],
                   gap=(s["final_val_loss"] - s["final_train_loss"]) if s["final_val_loss"] is not None else None,
                   val_acc=s["val_acc"], val_f1=s["val_macro_f1"], s_per_ep=s["time_per_epoch_s"],
                   mem_MB=s["peak_mem_MB"], diverged=s["diverged"])
        if base_mean is not None:
            row["ΔF1_vs_base"] = s["val_macro_f1"] - base_mean
            row["vượt 2σ?"] = "Có" if two_sigma and abs(row["ΔF1_vs_base"]) > two_sigma else "Không"
        rows.append(row)
    return pd.DataFrame(rows).set_index("exp_id").round(4)

### 2a. Chọn lr cho baseline bằng val
SGD + momentum, thử lr ∈ {0,01; 0,03; 0,1} (seed 1). Chọn lr có **val macro-F1** cao nhất. Lần chạy của lr được chọn chính là `base-s1`;
các lr còn lại giữ lại thành dòng nhóm `hparam` (thí nghiệm lr).

In [ ]:
LR_GRID = [0.01, 0.03, 0.1]
sweep = {}
for lr in LR_GRID:
    sweep[lr] = run({"exp_id": f"hp-sgdm-lr{lr}", "group": "hparam",
                     "description": f"Baseline nhưng lr={lr} (chọn lr cho baseline)", "lr": lr})
display(table([f"hp-sgdm-lr{lr}" for lr in LR_GRID]))

BASE_LR = max(LR_GRID, key=lambda lr: sweep[lr]["summary"]["val_macro_f1"])
print("lr baseline chọn theo val macro-F1:", BASE_LR)
BASE_CFG = {**DEFAULT_CFG, "lr": BASE_LR}

### 2b. Baseline với 3 seed → độ nhiễu

In [ ]:
SEEDS = [1, 2, 3]
for s in SEEDS:
    run({**BASE_CFG, "seed": s, "exp_id": f"base-s{s}", "group": "baseline",
         "description": f"Baseline M-base, seed {s}"}, verbose=(s == 1))

# lần chạy hp-sgdm-lr<BASE_LR> trùng cấu hình với base-s1 -> bỏ khỏi bảng để không lặp dòng
dup = f"{OUT_DIR}/results/hp-sgdm-lr{BASE_LR}.json"
if os.path.exists(dup):
    os.remove(dup)
    if os.path.exists(f"{OUT_DIR}/figures/hp-sgdm-lr{BASE_LR}.png"):
        os.remove(f"{OUT_DIR}/figures/hp-sgdm-lr{BASE_LR}.png")
RESULTS.pop(f"hp-sgdm-lr{BASE_LR}", None)

base_ids = [f"base-s{s}" for s in SEEDS]
f1s = np.array([RESULTS[i]["summary"]["val_macro_f1"] for i in base_ids])
accs = np.array([RESULTS[i]["summary"]["val_acc"] for i in base_ids])
BASE_MEAN, BASE_STD = f1s.mean(), f1s.std(ddof=1)
TWO_SIGMA = 2 * BASE_STD
print(f"val macro-F1 baseline: {BASE_MEAN:.4f} ± {BASE_STD:.4f}  (2σ = {TWO_SIGMA:.4f})")
print(f"val acc baseline     : {accs.mean():.4f} ± {accs.std(ddof=1):.4f}")
display(table(base_ids, BASE_MEAN, TWO_SIGMA))
plot_compare([RESULTS[i] for i in base_ids], ["train_loss", "val_loss", "val_macro_f1", "grad_norm"],
             f"{OUT_DIR}/figures/compare_baseline_seeds.png", "Baseline — 3 seed")

**Nhận xét baseline (điền sau khi chạy):** hình dạng đường cong (train/val loss còn giảm ở epoch 20? có tách nhau — quá khớp — không?),
val acc ___ so với mốc 0,4876, độ nhiễu giữa các seed σ = ___ ⇒ mọi chênh lệch val macro-F1 nhỏ hơn 2σ = ___ được coi là chưa kết luận được.

## Part 3 — Thí nghiệm
Mỗi thí nghiệm đổi **một** yếu tố so với baseline (seed 1, 20 epoch, cùng split). So sánh với **trung bình baseline** và ngưỡng **2σ**.

### Chủ đề 1 — Hàm mất mát: CE vs MSE (`loss-mse`)
**Dự đoán trước:** MSE trên logit với one-hot cho gradient tuyến tính theo sai số và không "phạt mạnh" dự đoán sai nặng như CE (gradient CE
theo logit = softmax − one-hot, không bão hoà). MSE cũng đối xử các lớp hiếm kém hơn ⇒ dự đoán val macro-F1 của MSE **thấp hơn** CE và hội tụ chậm hơn.
MSE: `F.mse_loss(logits, one_hot)` lấy trung bình trên B×7 phần tử, không hệ số 1/2. So sánh bằng accuracy/macro-F1, không so loss.

In [ ]:
run({**BASE_CFG, "exp_id": "loss-mse", "group": "loss", "description": "MSE trên logit vs one-hot thay cho CE", "loss": "mse"})
display(table(["base-s1", "loss-mse"], BASE_MEAN, TWO_SIGMA))
plot_compare([RESULTS["base-s1"], RESULTS["loss-mse"]], ["val_acc", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_loss.png", "CE vs MSE (so bằng metric, không so loss)")

**Đối chiếu sau khi chạy:** ...

### Chủ đề 2 — Bộ tối ưu hoá (mỗi bộ ≥ 2 lr, so ở lr tốt nhất của mỗi bộ)
- SGD (không momentum): lr ∈ {0,1; 0,3} (+ lr baseline đã có trong sweep nếu cần)
- SGD + momentum: dùng kết quả sweep ở Part 2 (0,01 / 0,03 / 0,1)
- Adam: lr ∈ {3e-4; 1e-3; 3e-3}
- AdamW (`weight_decay = 0,01`): lr ∈ {1e-3; 3e-3}

**Dự đoán trước:** SGD thuần cần lr lớn hơn SGD+momentum để đi cùng quãng đường (momentum ≈ nhân bước hiệu dụng 1/(1−μ) = 10 lần).
Adam chia bước theo độ lớn gradient từng tham số nên hội tụ nhanh ở vài epoch đầu; ở lr tốt nhất của mỗi bộ, Adam/AdamW sẽ cao hơn SGD+momentum
một chút. AdamW với wd = 0,01 gần như bằng Adam (mạng nhỏ, chưa quá khớp nhiều).

In [ ]:
OPT_GRID = {"sgd": [0.1, 0.3], "adam": [3e-4, 1e-3, 3e-3], "adamw": [1e-3, 3e-3]}
for opt, lrs in OPT_GRID.items():
    for lr in lrs:
        run({**BASE_CFG, "exp_id": f"opt-{opt}-lr{lr:g}", "group": "optimizer",
             "description": f"{opt} lr={lr:g}" + (" wd=0.01" if opt == "adamw" else ""),
             "optimizer": opt, "lr": lr, "weight_decay": 0.01 if opt == "adamw" else 0.0})

opt_runs = {"sgd": [f"opt-sgd-lr{lr:g}" for lr in OPT_GRID["sgd"]],
            "sgd_momentum": [f"hp-sgdm-lr{lr}" for lr in LR_GRID if lr != BASE_LR] + ["base-s1"],
            "adam": [f"opt-adam-lr{lr:g}" for lr in OPT_GRID["adam"]],
            "adamw": [f"opt-adamw-lr{lr:g}" for lr in OPT_GRID["adamw"]]}
display(table([i for ids in opt_runs.values() for i in ids], BASE_MEAN, TWO_SIGMA))

BEST_OPT = {k: max(ids, key=lambda i: RESULTS[i]["summary"]["val_macro_f1"]) for k, ids in opt_runs.items()}
print("lr tốt nhất của mỗi bộ (theo val macro-F1):", BEST_OPT)
plot_compare([RESULTS[i] for i in BEST_OPT.values()], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_optimizer.png", "Bộ tối ưu — mỗi bộ ở lr tốt nhất của nó")

**Đối chiếu sau khi chạy:** ...

### Chủ đề 3 — Hyper-parameter: lr (đã có từ sweep), batch size, độ rộng, độ sâu
- `hp-batch128`, `hp-batch2048` (giữ lr baseline) và `hp-batch2048-lrx4` (quy tắc tăng lr theo lô: lô ×4 ⇒ lr ×4)
- `hp-wide` (M-wide 512-256) và `hp-deep` (M-deep 256-128-64)

**Dự đoán trước:** cùng 20 epoch, batch 128 có gấp 4 lần số bước cập nhật so với 512 ⇒ val tốt hơn nhưng mỗi epoch chậm hơn;
batch 2048 chỉ có 1/4 số bước ⇒ kém hơn ở cùng lr, tăng lr ×4 bù lại phần lớn. M-wide nhiều tham số hơn ⇒ fit tốt hơn (dữ liệu lớn, chưa quá khớp)
nên val tăng; M-deep thay đổi ít.

In [ ]:
HP = [
    ("hp-batch128", "batch=128", {"batch": 128}),
    ("hp-batch2048", "batch=2048, giữ lr", {"batch": 2048}),
    ("hp-batch2048-lrx4", "batch=2048, lr×4 (quy tắc tăng lr theo lô)", {"batch": 2048, "lr": BASE_LR * 4}),
    ("hp-wide", "M-wide 54-512-256-7", {"hidden": (512, 256)}),
    ("hp-deep", "M-deep 54-256-128-64-7", {"hidden": (256, 128, 64)}),
]
for eid, desc, ch in HP:
    run({**BASE_CFG, "exp_id": eid, "group": "hparam", "description": desc, **ch})
hp_ids = [f"hp-sgdm-lr{lr}" for lr in LR_GRID if lr != BASE_LR] + [e for e, _, _ in HP]
display(table(["base-s1"] + hp_ids, BASE_MEAN, TWO_SIGMA))
plot_compare([RESULTS[i] for i in ["base-s1", "hp-batch128", "hp-batch2048", "hp-batch2048-lrx4"]],
             ["val_loss", "val_macro_f1", "epoch_time_s"], f"{OUT_DIR}/figures/compare_batch.png", "Batch size")
plot_compare([RESULTS[i] for i in ["base-s1", "hp-wide", "hp-deep"]], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_arch.png", "Kiến trúc: base / wide / deep")
plot_compare([RESULTS[i] for i in ["base-s1"] + [f"hp-sgdm-lr{lr}" for lr in LR_GRID if lr != BASE_LR]],
             ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_lr.png", "lr (SGD+momentum)")

**Đối chiếu sau khi chạy:** ...

### Chủ đề 4 — Dropout q ∈ {0,1; 0,3}
**Dự đoán trước:** nhìn khoảng cách val − train loss của baseline ở epoch 20: nếu nhỏ (mô hình chưa quá khớp) thì dropout chủ yếu làm giảm năng lực
và làm chậm hội tụ ⇒ val macro-F1 **không tăng**, q = 0,3 có thể giảm rõ. Khoảng cách train–val sẽ thu hẹp. (train loss đo ở eval mode nên so sánh được.)

In [ ]:
gap = RESULTS["base-s1"]["summary"]["final_val_loss"] - RESULTS["base-s1"]["summary"]["final_train_loss"]
print(f"khoảng cách val − train loss của baseline ở epoch cuối: {gap:.4f}")
for q in [0.1, 0.3]:
    run({**BASE_CFG, "exp_id": f"drop-{q}", "group": "dropout", "description": f"dropout q={q} sau mỗi ReLU ẩn", "dropout": q})
display(table(["base-s1", "drop-0.1", "drop-0.3"], BASE_MEAN, TWO_SIGMA))
plot_compare([RESULTS[i] for i in ["base-s1", "drop-0.1", "drop-0.3"]], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_dropout.png", "Dropout")

**Đối chiếu sau khi chạy:** ...

### Chủ đề 5 — Gradient clipping
1. Chọn `c` từ `grad_norm` của baseline: lấy **trung vị** của grad_norm trung bình theo epoch ⇒ clipping kích hoạt ở một phần đáng kể các bước.
2. Thí nghiệm phản chứng: lr ×10 so với baseline, **không clip** vs **có clip** cùng `c`.

**Dự đoán trước:** ở lr baseline, clip với c ≈ trung vị chỉ làm bước nhỏ đi một chút ⇒ khác biệt nằm trong nhiễu. Ở lr ×10, không clip sẽ dao động mạnh
(gai grad_norm, val loss nhảy, có thể NaN); có clip giới hạn độ dài bước cập nhật ≤ lr·c nên huấn luyện ổn định hơn.

In [ ]:
gn_base = np.array(RESULTS["base-s1"]["history"]["grad_norm"])
gn_max = np.array(RESULTS["base-s1"]["history"]["grad_norm_max"])
CLIP_C = float(np.round(np.median(gn_base), 2))
print(f"grad_norm baseline (TB/epoch): min {gn_base.min():.3f}, median {np.median(gn_base):.3f}, max {gn_base.max():.3f}; "
      f"max theo bước {gn_max.max():.3f}  ->  chọn c = {CLIP_C}")
HIGH_LR = BASE_LR * 10
run({**BASE_CFG, "exp_id": f"clip-{CLIP_C}", "group": "clipping", "description": f"clip c={CLIP_C} (trung vị grad_norm baseline)",
     "clip_norm": CLIP_C})
run({**BASE_CFG, "exp_id": "clip-none-highlr", "group": "clipping", "description": f"lr×10 = {HIGH_LR:g}, KHÔNG clip",
     "lr": HIGH_LR})
run({**BASE_CFG, "exp_id": f"clip-{CLIP_C}-highlr", "group": "clipping", "description": f"lr×10 = {HIGH_LR:g}, clip c={CLIP_C}",
     "lr": HIGH_LR, "clip_norm": CLIP_C})
clip_ids = ["base-s1", f"clip-{CLIP_C}", "clip-none-highlr", f"clip-{CLIP_C}-highlr"]
display(table(clip_ids, BASE_MEAN, TWO_SIGMA))
for i in clip_ids[1:]:
    if RESULTS[i]["cfg"]["clip_norm"]:
        print(i, "tỉ lệ bước bị clip theo epoch:", np.round(RESULTS[i]["history"]["clip_frac"], 2))
plot_compare([RESULTS[i] for i in clip_ids], ["val_loss", "val_macro_f1", "grad_norm_max"],
             f"{OUT_DIR}/figures/compare_clipping.png", f"Gradient clipping (c = {CLIP_C})")

**Đối chiếu sau khi chạy:** ...

### Chủ đề 6 — Mixed precision (FP16 + GradScaler, BF16)
**Dự đoán trước:** tham số vẫn FP32, autocast chỉ hạ độ chính xác của matmul ⇒ val macro-F1 nằm trong nhiễu so với FP32. Mạng rất nhỏ
(48k tham số, batch 512) nên thời gian bị chi phối bởi chi phí gọi kernel ⇒ có thể **không nhanh hơn**. FP16 cần GradScaler vì khoảng
biểu diễn hẹp (min normal ≈ 6e-5) ⇒ gradient nhỏ bị underflow; BF16 có cùng số mũ 8 bit với FP32 nên không cần. T4 không có phần cứng BF16.

In [ ]:
amp_ids = ["base-s1"]
if device == "cuda":
    run({**BASE_CFG, "exp_id": "amp-fp16", "group": "amp", "description": "autocast FP16 + GradScaler", "precision": "fp16"})
    amp_ids.append("amp-fp16")
    if torch.cuda.is_bf16_supported():
        run({**BASE_CFG, "exp_id": "amp-bf16", "group": "amp", "description": "autocast BF16 (không GradScaler)", "precision": "bf16"})
        amp_ids.append("amp-bf16")
    else:
        print("GPU không hỗ trợ BF16 -> bỏ qua amp-bf16 (ghi vào báo cáo)")
else:
    print("Không có GPU CUDA -> bỏ qua thí nghiệm mixed precision")
display(table(amp_ids, BASE_MEAN, TWO_SIGMA))
if len(amp_ids) > 1:
    plot_compare([RESULTS[i] for i in amp_ids], ["val_loss", "val_macro_f1", "epoch_time_s"],
                 f"{OUT_DIR}/figures/compare_amp.png", "Mixed precision")

**Đối chiếu sau khi chạy:** ...

### Chủ đề 7 — Khởi tạo tham số: zeros / normal(0,0.01²) / xavier_normal / default vs He
`xavier` dùng `xavier_normal_`: Var = 2/(n_in + n_out). `default` = `kaiming_uniform_(a=√5)` mặc định của nn.Linear (Var ≈ 1/(3·n_in)), **không phải He**.

**Dự đoán trước:** `zeros`: mọi nơ-ron trong một lớp giống hệt nhau, ReLU(0) = 0 nên gradient vào W1, W2 bằng 0 ⇒ chỉ bias lớp ra học được ⇒ mạng
chỉ học tỉ lệ lớp (≈ đoán đa số, macro-F1 ≈ 0,09). `normal(0,0.01²)`: kích hoạt co lại ~10 lần qua mỗi lớp ⇒ ban đầu học chậm nhưng mạng chỉ 3 lớp nên
vẫn hồi phục. xavier/default/He: với 3 lớp khác biệt nhỏ, có thể nằm trong nhiễu.

In [ ]:
INITS = ["zeros", "normal", "xavier", "default"]
xb_val = D["X_val"][:4096]
rows = []
for init in ["he"] + INITS:
    set_seed(1)
    mi = MLP(hidden=(256, 128), init=init).to(device)
    st = activation_stats(mi, xb_val)
    rows.append(dict(init=init, std_h1=st[0], std_h2=st[1], std_logits=st[2],
                     step0_loss=evaluate(mi, D["X_val"], D["y_val"])["loss"]))
display(pd.DataFrame(rows).set_index("init").round(4))

for init in INITS:
    run({**BASE_CFG, "exp_id": f"init-{init}", "group": "init", "description": f"khởi tạo {init} thay cho He", "init": init})
init_ids = ["base-s1"] + [f"init-{i}" for i in INITS]
display(table(init_ids, BASE_MEAN, TWO_SIGMA))
plot_compare([RESULTS[i] for i in init_ids], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_init.png", "Khởi tạo tham số")

**Đối chiếu sau khi chạy:** ...

### Kết hợp → cấu hình cuối cùng (chọn **chỉ bằng val**)
Quy tắc chọn (viết trước khi chạy, không nhìn eval):
1. Bộ tối ưu + lr: lấy lần chạy có val macro-F1 cao nhất trong nhóm optimizer (kể cả SGD+momentum baseline).
2. Kiến trúc: M-wide nếu `hp-wide` hơn baseline trên val, ngược lại M-base.
3. Dropout: chỉ dùng nếu `drop-*` tốt nhất hơn baseline **vượt 2σ**.
4. Huấn luyện lâu hơn: 40 epoch + cosine decay lr (đường cong baseline thường còn đi lên ở epoch 20).

Chạy cấu hình này với 2 seed. Cấu hình nộp = lần chạy có **val macro-F1** cao nhất trong số {các thí nghiệm đơn lẻ, final-s1, final-s2}.

In [ ]:
opt_pool = [i for ids in opt_runs.values() for i in ids]
best_opt_id = max(opt_pool, key=lambda i: RESULTS[i]["summary"]["val_macro_f1"])
bo = RESULTS[best_opt_id]["cfg"]
use_wide = RESULTS["hp-wide"]["summary"]["val_macro_f1"] > RESULTS["base-s1"]["summary"]["val_macro_f1"]
best_drop = max(["drop-0.1", "drop-0.3"], key=lambda i: RESULTS[i]["summary"]["val_macro_f1"])
use_drop = RESULTS[best_drop]["summary"]["val_macro_f1"] - BASE_MEAN > TWO_SIGMA

FINAL_CFG = {**BASE_CFG, "group": "final",
             "optimizer": bo["optimizer"], "lr": bo["lr"], "weight_decay": bo["weight_decay"],
             "hidden": (512, 256) if use_wide else (256, 128),
             "dropout": RESULTS[best_drop]["cfg"]["dropout"] if use_drop else 0.0,
             "epochs": 40, "scheduler": "cosine"}
FINAL_CFG["description"] = (f"Kết hợp: {bo['optimizer']} lr={bo['lr']:g} wd={bo['weight_decay']}, "
                            f"hidden={'-'.join(map(str, FINAL_CFG['hidden']))}, dropout={FINAL_CFG['dropout']}, 40 epoch + cosine")
print("best optimizer run:", best_opt_id, "| wide:", use_wide, "| dropout:", use_drop)
print(FINAL_CFG["description"])

for s in [1, 2]:
    run({**FINAL_CFG, "seed": s, "exp_id": f"final-s{s}"}, verbose=(s == 1))
final_ids = ["final-s1", "final-s2"]
display(table(["base-s1"] + final_ids, BASE_MEAN, TWO_SIGMA))
ff1 = np.array([RESULTS[i]["summary"]["val_macro_f1"] for i in final_ids])
print(f"final val macro-F1: {ff1.mean():.4f} ± {ff1.std(ddof=1):.4f}  vs baseline {BASE_MEAN:.4f} ± {BASE_STD:.4f}")
plot_compare([RESULTS[i] for i in base_ids + final_ids], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_final.png", "Baseline (3 seed) vs cấu hình cuối (2 seed)")

# chọn mô hình nộp CHỈ theo val
candidates = [i for i in RESULTS if not RESULTS[i]["summary"]["diverged"]]
SUBMIT_ID = max(candidates, key=lambda i: RESULTS[i]["summary"]["val_macro_f1"])
print("Mô hình nộp (val macro-F1 cao nhất):", SUBMIT_ID, f"{RESULTS[SUBMIT_ID]['summary']['val_macro_f1']:.4f}")

**Nhận xét cấu hình cuối (điền sau khi chạy):** ...

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
Chỉ chạy cho **baseline (`base-s1`)** và **mô hình nộp (`SUBMIT_ID`)**, sau khi đã chọn cấu hình bằng val. Không chỉnh cấu hình sau khi thấy điểm eval.
`final_eval` nạp trọng số ở epoch có val loss thấp nhất, dự đoán toàn bộ eval, ghi CSV và chạy đúng `scripts/evaluate.py`.

In [ ]:
EVAL = {}
r_base = ensure_state("base-s1")
EVAL["base-s1"] = final_eval(r_base["cfg"], r_base, D, f"{OUT_DIR}/extra/predictions_eval_baseline.csv",
                             repo_root=REPO_ROOT, out_json=f"{OUT_DIR}/extra/eval_result_baseline.json")

r_sub = ensure_state(SUBMIT_ID)
EVAL[SUBMIT_ID] = final_eval(r_sub["cfg"], r_sub, D, f"{OUT_DIR}/predictions_eval.csv",
                             repo_root=REPO_ROOT, out_json=f"{OUT_DIR}/eval_result.json")
for k, v in EVAL.items():
    print(f"{k:14s} eval accuracy = {v['accuracy']:.4f}   eval macro-F1 = {v['macro_f1']:.4f}")
print(f"cải thiện macro-F1 (eval) = {EVAL[SUBMIT_ID]['macro_f1'] - EVAL['base-s1']['macro_f1']:+.4f}")

In [ ]:
# Phân tích lỗi theo lớp trên eval (mô hình nộp)
with open(f"{OUT_DIR}/eval_result.json", encoding="utf-8") as f:
    ER = json.load(f)
n_train_cls = np.bincount(D["y_tr"].cpu().numpy(), minlength=7)
pc = pd.DataFrame(ER["per_class"]).set_index("cls")
pc["n_train"] = n_train_cls
display(pc.round(4))
cm = np.array(ER["confusion_matrix"])
cm_norm = cm / cm.sum(1, keepdims=True)
display(pd.DataFrame(cm, index=[f"thật {c}" for c in range(7)], columns=[f"dự đoán {c}" for c in range(7)]))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_norm, cmap="Blues")
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xlabel("dự đoán"); ax.set_ylabel("nhãn thật"); ax.set_title(f"Ma trận nhầm lẫn chuẩn hoá theo hàng — {SUBMIT_ID} (eval)")
plt.colorbar(im); plt.tight_layout(); plt.show()

worst = int(pc["f1"].idxmin())
off = cm[worst].copy(); off[worst] = 0
print(f"lớp khó nhất: {worst} (F1 = {pc.loc[worst, 'f1']:.4f}, support eval = {pc.loc[worst, 'support']}, n_train = {n_train_cls[worst]}); "
      f"hay bị nhầm sang lớp {int(off.argmax())} ({off.max()} mẫu, {off.max() / cm[worst].sum():.1%})")

**Phân tích lỗi (điền sau khi chạy):** lớp khó nhất ___ (F1 ___), bị nhầm chủ yếu sang lớp ___. Giải thích bằng dữ liệu: số mẫu train của lớp,
lớp đó có cùng vùng độ cao/Wilderness_Area với lớp bị nhầm (ví dụ lớp 0 Spruce/Fir và lớp 1 Lodgepole Pine chồng lấn mạnh về Elevation) ...

In [ ]:
# Bảng experiments.xlsx (từ results/*.json) + sheet Seeds + nhận xét Summary
all_res = load_results(f"{OUT_DIR}/results")
rows = [to_row(r, EVAL.get(r["cfg"]["exp_id"])) for r in all_res]

def group_note(g):
    ids = [r["cfg"]["exp_id"] for r in all_res if r["cfg"]["group"] == g]
    if not ids:
        return None
    best = max(ids, key=lambda i: RESULTS[i]["summary"]["val_macro_f1"])
    d = RESULTS[best]["summary"]["val_macro_f1"] - BASE_MEAN
    return (f"Tốt nhất: {best} (val F1 {RESULTS[best]['summary']['val_macro_f1']:.4f}, Δ vs TB baseline {d:+.4f}, "
            f"2σ = {TWO_SIGMA:.4f} -> {'vượt nhiễu' if abs(d) > TWO_SIGMA else 'trong nhiễu'})")

notes = {g: group_note(g) for g in ["baseline", "loss", "optimizer", "hparam", "dropout", "clipping", "amp", "init", "final"]}
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx",
           seed_ids=base_ids, summary_notes={k: v for k, v in notes.items() if v})
print("đã ghi experiments.xlsx với", len(rows), "dòng")

# kiểm tra: mỗi dòng có đúng một ảnh figures/<exp_id>.png
ids = {r["exp_id"] for r in rows}
pngs = {os.path.splitext(f)[0] for f in os.listdir(f"{OUT_DIR}/figures") if not f.startswith("compare_")}
print("thiếu ảnh:", ids - pngs or "không", "| ảnh thừa:", pngs - ids or "không")
display(pd.DataFrame(rows).set_index("exp_id")[["group", "optimizer", "lr", "val_acc", "val_macro_f1", "eval_acc", "eval_macro_f1"]])